# Введение в MapReduce модель на Python


In [2]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [3]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [4]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [5]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [6]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [7]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [8]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [9]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [10]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [11]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [12]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [13]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных. 

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [14]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*
 
mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL 

In [15]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str
    
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)
 
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication 

In [16]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])
 
def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])
      
output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(1.5814677307189082)),
 (1, np.float64(1.5814677307189082)),
 (2, np.float64(1.5814677307189082)),
 (3, np.float64(1.5814677307189082)),
 (4, np.float64(1.5814677307189082))]

## Inverted index 

In [17]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)
      
def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)
 
def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [18]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [19]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()
      
def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]
 
def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers
  
def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)
  
  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*
 
flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount 

In [20]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)
      
  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)
  
# try to set COMBINER=REDUCER and look at the number of values sent over the network 
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None) 
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('is', 18), ('it', 18), ('what', 10)]),
 (1, [('a', 2), ('banana', 2)])]

## TeraSort

In [21]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for value in split:
        yield (value, None)
      
  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])
    
def MAP(value:int, _):
  yield (value, None)
  
def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)
  
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.03372429543813682)),
   (None, np.float64(0.036239945656642236)),
   (None, np.float64(0.07047276677121084)),
   (None, np.float64(0.1487301017905217)),
   (None, np.float64(0.15715428490617656)),
   (None, np.float64(0.19809727317162085)),
   (None, np.float64(0.2914453871231678)),
   (None, np.float64(0.3135240050836613)),
   (None, np.float64(0.3431337175059527)),
   (None, np.float64(0.3874552229265855)),
   (None, np.float64(0.3931481978392799)),
   (None, np.float64(0.4321841936955054)),
   (None, np.float64(0.4362301946347503)),
   (None, np.float64(0.43673981469184664)),
   (None, np.float64(0.47707970365311325)),
   (None, np.float64(0.4938111432879798)),
   (None, np.float64(0.4965645092880796))]),
 (1,
  [(None, np.float64(0.5261390297819541)),
   (None, np.float64(0.5464624305813894)),
   (None, np.float64(0.5768094508012448)),
   (None, np.float64(0.6140693180257967)),
   (None, np.float64(0.6373852146898855)),
   (None, np.float64(0.66527414379

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [23]:
def RECORDREADER():
    l = [10, 5, 100, 2, 50]
    for v in l:
        yield (None, v)

def MAP(key, value):
    yield ('max_key', value)

def REDUCE(key, values):
    yield (key, max(values))

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('max_key', 100)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [ ]:
def RECORDREADER():
    l = [10, 20, 30, 40, 50]
    for v in l:
        yield (None, v)

def MAP(key, value):
    yield ('mean_key', (value, 1))

def REDUCE(key, values):
    total_sum = 0
    total_count = 0
    for val, count in values:
        total_sum += val
        total_count += count
    
    if total_count > 0:
        yield (key, total_sum / total_count)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('mean_key', 30.0)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [ ]:

def groupbykey_sorted(iterable):

    sorted_data = sorted(iterable, key=lambda x: x[0])
    
    result = []
    if not sorted_data:
        return result

    current_key = sorted_data[0][0]
    current_group = []

    for key, value in sorted_data:
        if key == current_key:
            current_group.append(value)
        else:
            yield (current_key, current_group)
            current_key = key
            current_group = [value]
    
    if current_group:
        yield (current_key, current_group)

def MapReduceSorted(RECORDREADER, MAP, REDUCE):
    return flatten(map(lambda x: REDUCE(*x), groupbykey_sorted(flatten(map(lambda x: MAP(*x), RECORDREADER())))))


d1 = "foo bar foo"
def RR_TEST():
    yield (0, d1)
def MAP_TEST(k, v):
    for w in v.split(): yield (w, 1)
def RED_TEST(k, v):
    yield (k, sum(v))

print(list(MapReduceSorted(RR_TEST, MAP_TEST, RED_TEST)))

[('bar', 1), ('foo', 2)]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [ ]:
def RECORDREADER():
    l = [1, 2, 3, 1, 2, 4, 5]
    for v in l:
        yield (None, v)

def MAP(key, value):
    yield (value, None)

def REDUCE(key, values):
    yield key

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[1, 2, 3, 4, 5]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [ ]:
relation_R = [
    (5, 'Alice'),
    (15, 'Bob'),
    (20, 'Charlie'),
    (2, 'David')
]

def RECORDREADER():
    for row in relation_R:
        yield (None, row)

def MAP(_, row):
    if row[0] > 10:
        yield (row, row)

def REDUCE(key, values):
    yield (key, key)

print("Selection Result:", list(MapReduce(RECORDREADER, MAP, REDUCE)))

Selection Result: [((15, 'Bob'), (15, 'Bob')), ((20, 'Charlie'), (20, 'Charlie'))]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [ ]:
relation_R = [
    (1, 'Alice'),
    (2, 'Bob'),
    (3, 'Alice'), 
    (4, 'Charlie')
]

def RECORDREADER():
    for row in relation_R:
        yield (None, row)

def MAP(_, row):
    t_prime = (row[1],) 
    yield (t_prime, t_prime)

def REDUCE(key, values):
    yield (key, key)

print("Projection Result:", list(MapReduce(RECORDREADER, MAP, REDUCE)))

Projection Result: [(('Alice',), ('Alice',)), (('Bob',), ('Bob',)), (('Charlie',), ('Charlie',))]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [ ]:
R = [(1, 'A'), (2, 'B')]
S = [(2, 'B'), (3, 'C')]

def RECORDREADER():
    for row in R: yield (None, row)
    for row in S: yield (None, row)

def MAP(_, row):
    yield (row, row)

def REDUCE(key, values):
    yield (key, key)

print("Union Result:", list(MapReduce(RECORDREADER, MAP, REDUCE)))

Union Result: [((1, 'A'), (1, 'A')), ((2, 'B'), (2, 'B')), ((3, 'C'), (3, 'C'))]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [ ]:
R = [(1, 'A'), (2, 'B'), (4, 'D')]
S = [(2, 'B'), (3, 'C'), (4, 'D')]

def RECORDREADER():
    for row in R: yield (None, row)
    for row in S: yield (None, row)

def MAP(_, row):
    yield (row, row)

def REDUCE(key, values):
    if len(values) == 2:
        yield (key, key)

print("Intersection Result:", list(MapReduce(RECORDREADER, MAP, REDUCE)))

Intersection Result: [((2, 'B'), (2, 'B')), ((4, 'D'), (4, 'D'))]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [ ]:
R = [(1, 'A'), (2, 'B'), (3, 'C')]
S = [(2, 'B'), (4, 'D')]

def RECORDREADER():
    for row in R: yield (None, (row, 'R'))
    for row in S: yield (None, (row, 'S'))

def MAP(_, value):
    row, relation_name = value
    yield (row, relation_name)

def REDUCE(key, relations):
    if 'R' in relations and 'S' not in relations:
        yield (key, key)

print("Difference Result:", list(MapReduce(RECORDREADER, MAP, REDUCE)))

Difference Result: [((1, 'A'), (1, 'A')), ((3, 'C'), (3, 'C'))]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [ ]:
R = [('Alice', 10), ('Bob', 20), ('Charlie', 10)]
S = [(10, 'IT'), (20, 'Sales')]

def RECORDREADER():
    for row in R: yield (None, (row, 'R'))
    for row in S: yield (None, (row, 'S'))

def MAP(_, value):
    row, relation = value
    if relation == 'R':
        yield (row[1], ('R', row[0]))
    else:
        yield (row[0], ('S', row[1]))

def REDUCE(key, values):
    r_rows = [v[1] for v in values if v[0] == 'R']
    s_rows = [v[1] for v in values if v[0] == 'S']
    for a in r_rows:
        for c in s_rows:
            yield (None, (a, key, c))

print("Natural Join Result:", list(MapReduce(RECORDREADER, MAP, REDUCE)))

Natural Join Result: [(None, ('Alice', 10, 'IT')), (None, ('Charlie', 10, 'IT')), (None, ('Bob', 20, 'Sales'))]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [ ]:
data = [('IT', 100), ('Sales', 150), ('IT', 120), ('Sales', 200)]

def RECORDREADER():
    for row in data: yield (None, row)

def MAP(_, row):
    yield (row[0], row[1])

def REDUCE(key, values):
    yield (key, sum(values))

print("Grouping Result:", list(MapReduce(RECORDREADER, MAP, REDUCE)))

Grouping Result: [('IT', 220), ('Sales', 350)]


# 

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [ ]:
import numpy as np

matrix_data = [
    (0, 0, 1), (0, 1, 2),
    (1, 0, 3), (1, 1, 4),
    (2, 0, 5), (2, 1, 6)
]


vector_data = [
    (0, 10),
    (1, 20)
]

def RECORDREADER():
    for (i, j, m_val) in matrix_data:
        yield (None, ('M', i, j, m_val))
    for (j, v_val) in vector_data:
        yield (None, ('V', j, v_val))

def MAP(_, value):
    tag = value[0]
    if tag == 'M':
        yield (value[2], ('M', value[1], value[3]))
    else:
        yield (value[1], ('V', value[2]))

def REDUCE(j, values):
    v_val = None
    m_entries = []
    
    for item in values:
        if item[0] == 'V':
            v_val = item[1]
        else:
            m_entries.append((item[1], item[2]))

    if v_val is not None:
        for (i, m_val) in m_entries:
            yield (i, m_val * v_val)


intermediate = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Intermediate products (i, m_ij * v_j):", intermediate)

final_vector = {}
for i, val in intermediate:
    final_vector[i] = final_vector.get(i, 0) + val
print("Final Matrix-Vector Product:", final_vector)

Intermediate products (i, m_ij * v_j): [(0, 10), (1, 30), (2, 50), (0, 40), (1, 80), (2, 120)]
Final Matrix-Vector Product: {0: 50, 1: 110, 2: 170}


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$. 





In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [25]:
import numpy as np

I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) 
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  
  for i in range(small_mat.shape[0]):
    yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
  yield (key, sum(values))


Проверьте своё решение

In [26]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat) 
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [27]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [28]:
I = 2
J = 3
K = 2

mat_M = np.random.rand(I, J)
mat_N = np.random.rand(J, K)

def RECORDREADER():
    for i in range(mat_M.shape[0]):
        for j in range(mat_M.shape[1]):
            yield (('M', i, j), mat_M[i,j])
    
    for j in range(mat_N.shape[0]):
        for k in range(mat_N.shape[1]):
            yield (('N', j, k), mat_N[j,k])

def MAP(key, value):
    matrix_name, row, col = key
    
    if matrix_name == 'M':
        i, j = row, col
        for k in range(K):
            yield ((i, k), ('M', j, value))
            
    elif matrix_name == 'N':
        j, k = row, col
        for i in range(I):
            yield ((i, k), ('N', j, value))

def REDUCE(key, values):
    vals_m = {}
    vals_n = {}
    
    for origin, j, val in values:
        if origin == 'M':
            vals_m[j] = val
        else:
            vals_n[j] = val
    result = 0
    for j in range(J):
        result += vals_m.get(j, 0) * vals_n.get(j, 0)
        
    yield (key, result)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))

result_dict = {k: v for k, v in output}
result_mat = np.zeros((I, K))
for i in range(I):
    for k in range(K):
        result_mat[i, k] = result_dict.get((i, k), 0)

print("Результат совпадает с numpy:", np.allclose(result_mat, np.matmul(mat_M, mat_N)))
print(result_mat)

Результат совпадает с numpy: True
[[1.74131884 0.65836118]
 [1.48603513 0.51652752]]


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER. 

In [29]:
def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]
 
def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers
  
def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)
  
  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs


maps = 2
reducers = 2

def INPUTFORMAT():
    def reader_M():
        for i in range(mat_M.shape[0]):
            for j in range(mat_M.shape[1]):
                yield (('M', i, j), mat_M[i,j])
    
    def reader_N():
        for j in range(mat_N.shape[0]):
            for k in range(mat_N.shape[1]):
                yield (('N', j, k), mat_N[j,k])
                
    yield reader_M()
    yield reader_N()


print("--- Distributed Matrix Multiplication ---")
distributed_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)

final_results = []
for part_id, iterator in distributed_output:
    for item in iterator:
        final_results.append(item)

print("Количество элементов результата:", len(final_results))
print("Пример элемента:", final_results[0])

--- Distributed Matrix Multiplication ---
24 key-value pairs were sent over a network.
Количество элементов результата: 4
Пример элемента: ((0, 1), np.float64(0.6583611838877479))


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [30]:
def INPUTFORMAT_RANDOM():
    all_elements = []
    for i in range(mat_M.shape[0]):
        for j in range(mat_M.shape[1]):
            all_elements.append( (('M', i, j), mat_M[i,j]) )
    for j in range(mat_N.shape[0]):
        for k in range(mat_N.shape[1]):
            all_elements.append( (('N', j, k), mat_N[j,k]) )
            
    np.random.shuffle(all_elements)
    
    num_splits = 4
    split_size = int(np.ceil(len(all_elements) / num_splits))
    
    for i in range(0, len(all_elements), split_size):
        chunk = all_elements[i : i + split_size]
        def reader_chunk(c=chunk):
            for item in c:
                yield item
        yield reader_chunk()

print("\n--- Randomized/Sharded Matrix Multiplication ---")
random_output = MapReduceDistributed(INPUTFORMAT_RANDOM, MAP, REDUCE)

final_results_random = []
for part_id, iterator in random_output:
    for item in iterator:
        final_results_random.append(item)

res_dict = {k: v for k, v in final_results_random}
check_mat = np.zeros((I, K))
for i in range(I):
    for k in range(K):
        check_mat[i, k] = res_dict.get((i, k), 0)

print("Результат совпадает при случайном разбиении:", np.allclose(check_mat, np.matmul(mat_M, mat_N)))


--- Randomized/Sharded Matrix Multiplication ---
24 key-value pairs were sent over a network.
Результат совпадает при случайном разбиении: True
